In [0]:
import tarfile
import os

tar_path     = "/Volumes/msbabigdata/spark/bda_project/yelp_dataset.tar"
extract_path = "/Volumes/msbabigdata/spark/bda_project/yelp_dataset/"

os.makedirs(extract_path, exist_ok=True)

with tarfile.open(tar_path, 'r:*') as tar:
    tar.extractall(extract_path)

os.listdir(extract_path)

In [0]:
BASE_PATH = "/Volumes/msbabigdata/spark/bda_project/yelp_dataset/"

business_df = spark.read.json(BASE_PATH + "yelp_academic_dataset_business.json")
review_df   = spark.read.json(BASE_PATH + "yelp_academic_dataset_review.json")
user_df     = spark.read.json(BASE_PATH + "yelp_academic_dataset_user.json")
checkin_df  = spark.read.json(BASE_PATH + "yelp_academic_dataset_checkin.json")
tip_df      = spark.read.json(BASE_PATH + "yelp_academic_dataset_tip.json")

print("business:", business_df.count())
print("reviews :", review_df.count())
print("users   :", user_df.count())
print("checkins:", checkin_df.count())
print("tips    :", tip_df.count())

In [0]:
business_df.createOrReplaceTempView("businesses")
review_df.createOrReplaceTempView("reviews")
user_df.createOrReplaceTempView("users")
checkin_df.createOrReplaceTempView("checkins")
tip_df.createOrReplaceTempView("tips")

print("All views registered.")

In [0]:
%sql
SELECT 
    city,
    state,
    COUNT(*)            AS business_count,
    SUM(review_count)   AS total_reviews,
    ROUND(AVG(stars),2) AS avg_stars
FROM businesses
GROUP BY city, state
ORDER BY business_count DESC
LIMIT 30;

In [0]:
%sql
SELECT
    CASE
        WHEN review_count < 10   THEN '1_0-9'
        WHEN review_count < 25   THEN '2_10-24'
        WHEN review_count < 50   THEN '3_25-49'
        WHEN review_count < 100  THEN '4_50-99'
        WHEN review_count < 250  THEN '5_100-249'
        WHEN review_count < 500  THEN '6_250-499'
        ELSE                          '7_500+'
    END                             AS review_bucket,
    COUNT(*)                        AS business_count,
    ROUND(COUNT(*) * 100.0 
        / SUM(COUNT(*)) OVER(), 2)  AS pct_of_total
FROM businesses
WHERE city = 'Philadelphia'
GROUP BY review_bucket
ORDER BY review_bucket;  -- the number prefix 1_,2_,3_... forces correct order

In [0]:
%sql
SELECT
    threshold,
    COUNT(*) AS businesses_remaining
FROM businesses
CROSS JOIN (
    SELECT explode(array(10, 25, 50, 100, 200)) AS threshold
)
WHERE city = 'Philadelphia'    -- swap after Cell A
  AND review_count >= threshold
GROUP BY threshold
ORDER BY threshold;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW businesses_filtered AS
SELECT *
FROM businesses
WHERE city        = 'Philadelphia'   -- change to your city
  AND review_count >= 50;            -- change to your threshold

SELECT COUNT(*) AS business_count FROM businesses_filtered;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW merged AS
SELECT
    -- Review fields
    r.review_id,
    r.date              AS review_date,
    r.stars             AS review_stars,
    r.text              AS review_text,
    r.useful,
    r.funny,
    r.cool,

    -- Business fields
    b.business_id,
    b.name              AS business_name,
    b.address,
    b.city,
    b.state,
    b.postal_code,
    b.stars             AS business_stars,
    b.review_count,
    b.categories,

    -- User fields
    u.user_id,
    u.name              AS user_name,
    u.review_count      AS user_review_count,
    u.average_stars     AS user_avg_stars,
    u.fans,
    u.yelping_since

FROM reviews r
INNER JOIN businesses_filtered b ON r.business_id = b.business_id
LEFT  JOIN users u               ON r.user_id     = u.user_id;

SELECT COUNT(*) AS total_merged_rows FROM merged;

In [0]:
%sql
SELECT * FROM merged LIMIT 20;